# Partitioning and Bucketing
---
Partitioning and Bucketing are techniques to organize data while writing for faster queries during reads (because write once read many). Partitioning splits data into separate folders based on column or group of columnar values becoming its own directory or disk. (partition pruning)

###Databricks ships with sample datasets you can use right away. The best ones for partitioning/bucketing practice are in the samples.tpch catalog (TPC-H benchmark data) and the samples.nyctaxi catalog. These are real-world-sized tables with natural partition/join keys.


|Aspect|Partitioning|Bucketing|
|---|---|---|
|Storage|Separate directories|Fixed number of files|
|Best for columns with|Low-to-medium cardinality (e.g., year, country)|High cardinality (e.g., user_id)|
|Main benefit|Skips reading irrelevant data|Avoids shuffles in joins/aggregations|
|Risk|Too many small files if cardinality is high|Wrong bucket count hurts performance|

In [0]:
spark.sql("SHOW CATALOGS").show()

In [0]:
spark.sql("SHOW SCHEMAS IN samples").show()

In [0]:
spark.sql("SHOW TABLES IN samples.tpch").show()

In [0]:
spark.sql("SHOW TABLES IN samples.nyctaxi").show()


###We'll find tables like samples.tpch.orders, samples.tpch.lineitem, samples.tpch.customer, and samples.nyctaxi.trips. These are read-only, so we'll copy them into our own database to experiment.


In [0]:
from pyspark.sql.functions import col, year, month, dayofmonth
import time

spark.sql("CREATE DATABASE IF NOT EXISTS partitioning_lab")
spark.sql("USE partitioning_lab")

# Peek at the data
spark.sql("SELECT * FROM samples.tpch.orders LIMIT 5").show()
spark.sql("SELECT * FROM samples.nyctaxi.trips LIMIT 5").show()

# Check sizes
print("Orders rows:", spark.table("samples.tpch.orders").count())
print("Lineitem rows:", spark.table("samples.tpch.lineitem").count())
print("Taxi trips rows:", spark.table("samples.nyctaxi.trips").count())


##Baseline — Unpartitioned Copy
Copy the TPC-H orders table into your database without any partitioning. This is your control group.

In [0]:
spark.sql("DROP TABLE IF EXISTS orders_raw")

spark.sql("""
    CREATE TABLE orders_raw AS
    SELECT * FROM samples.tpch.orders
""")

# Time a date-filtered query
start = time.time()
spark.sql("""
    SELECT COUNT(*), SUM(o_totalprice) 
    FROM orders_raw 
    WHERE o_orderdate BETWEEN '1995-01-01' AND '1995-12-31'
""").show()
print(f"Unpartitioned: {time.time() - start:.2f}s")

spark.sql("""
    SELECT COUNT(*) FROM orders_raw 
    WHERE o_orderdate BETWEEN '1995-01-01' AND '1995-12-31'
""").explain(True)
# Notice: full table scan, no PartitionFilters

##Partition by Year
The o_orderdate column is a natural partition candidate when extracted to year level.

In [0]:
spark.sql("DROP TABLE IF EXISTS orders_by_year")

orders = spark.table("samples.tpch.orders") \
    .withColumn("order_year", year(col("o_orderdate")))

orders.write.mode("overwrite") \
    .partitionBy("order_year") \
    .saveAsTable("orders_by_year")

# Inspect partitions
spark.sql("SHOW PARTITIONS orders_by_year").show()

# Same query, now with partition column
start = time.time()
spark.sql("""
    SELECT COUNT(*), SUM(o_totalprice) 
    FROM orders_by_year 
    WHERE order_year = 1995
""").show()
print(f"Partitioned by year: {time.time() - start:.2f}s")

spark.sql("SELECT COUNT(*) FROM orders_by_year WHERE order_year = 1995").explain(True)
# Look for: PartitionFilters: [(order_year = 1995)]

##Multi-Column Partitioning with NYC Taxi Data
The taxi dataset has natural year/month partition candidates.

In [0]:
spark.sql("DROP TABLE IF EXISTS taxi_partitioned")

taxi = spark.table("samples.nyctaxi.trips") \
    .withColumn("trip_year", year(col("tpep_pickup_datetime"))) \
    .withColumn("trip_month", month(col("tpep_pickup_datetime")))

# Check distribution before partitioning
taxi.groupBy("trip_year", "trip_month").count().orderBy("trip_year", "trip_month").show()

taxi.write.mode("overwrite") \
    .partitionBy("trip_year", "trip_month") \
    .saveAsTable("taxi_partitioned")

# Query a specific month
start = time.time()
spark.sql("""
    SELECT COUNT(*), AVG(fare_amount), AVG(trip_distance)
    FROM taxi_partitioned 
    WHERE trip_year = 2016 AND trip_month = 2
""").show()
print(f"Multi-partition query: {time.time() - start:.2f}s")

##Bucketing for the TPC-H Join
The classic TPC-H query joins orders with lineitem on o_orderkey = l_orderkey. This is a perfect bucketing scenario.

In [0]:
spark.sql("DROP TABLE IF EXISTS orders_bucketed")
spark.sql("DROP TABLE IF EXISTS lineitem_bucketed")

# Bucket both tables on the join key with the SAME number of buckets
spark.table("samples.tpch.orders").write.mode("overwrite") \
    .format("parquet") \
    .bucketBy(16, "o_orderkey") \
    .sortBy("o_orderkey") \
    .saveAsTable("orders_bucketed")

spark.table("samples.tpch.lineitem").write.mode("overwrite") \
    .format("parquet") \
    .bucketBy(16, "l_orderkey") \
    .sortBy("l_orderkey") \
    .saveAsTable("lineitem_bucketed")

# Verify bucketing
spark.sql("DESCRIBE EXTENDED orders_bucketed").show(50, truncate=False)

##Prove the Shuffle is Gone

In [0]:
# Bucketed join
bucketed_query = spark.sql("""
    SELECT o.o_orderkey, o.o_orderdate, l.l_partkey, l.l_quantity
    FROM orders_bucketed o
    JOIN lineitem_bucketed l ON o.o_orderkey = l.l_orderkey
    WHERE o.o_orderstatus = 'F'
""")
bucketed_query.explain()
# You should NOT see "Exchange hashpartitioning" between the scan and the join

# Compare to a join on the original (non-bucketed) tables
unbucketed_query = spark.sql("""
    SELECT o.o_orderkey, o.o_orderdate, l.l_partkey, l.l_quantity
    FROM samples.tpch.orders o
    JOIN samples.tpch.lineitem l ON o.o_orderkey = l.l_orderkey
    WHERE o.o_orderstatus = 'F'
""")
unbucketed_query.explain()
# You WILL see "Exchange hashpartitioning(o_orderkey, ...)" — that's the shuffle bucketing avoids

##The Hybrid — Partitioning + Bucketing
The sweet spot for many real workloads: partition by date, bucket by join key.

In [0]:
spark.sql("DROP TABLE IF EXISTS orders_hybrid")

orders = spark.table("samples.tpch.orders") \
    .withColumn("order_year", year(col("o_orderdate")))

orders.write.mode("overwrite") \
    .format("parquet") \
    .partitionBy("order_year") \
    .bucketBy(8, "o_orderkey") \
    .sortBy("o_orderkey") \
    .saveAsTable("orders_hybrid")

# Query that benefits from both: filter by year, join on orderkey
spark.sql("""
    SELECT COUNT(*) 
    FROM orders_hybrid o
    JOIN lineitem_bucketed l ON o.o_orderkey = l.l_orderkey
    WHERE o.order_year = 1995
""").explain(True)

##Liquid Clustering on TPC-H
Compare liquid clustering against traditional partitioning:


In [0]:
spark.sql("DROP TABLE IF EXISTS orders_liquid")

spark.sql("""
    CREATE TABLE orders_liquid (
        o_orderkey BIGINT,
        o_custkey BIGINT,
        o_orderstatus STRING,
        o_totalprice DECIMAL(18,2),
        o_orderdate DATE,
        o_orderpriority STRING,
        o_clerk STRING,
        o_shippriority INT,
        o_comment STRING
    )
    USING DELTA
    CLUSTER BY (o_orderdate, o_orderstatus)
""")

spark.sql("""
    INSERT INTO orders_liquid
    SELECT * FROM samples.tpch.orders
""")

spark.sql("OPTIMIZE orders_liquid")

# Query
start = time.time()
spark.sql("""
    SELECT COUNT(*), SUM(o_totalprice)
    FROM orders_liquid
    WHERE o_orderdate BETWEEN '1995-01-01' AND '1995-12-31'
      AND o_orderstatus = 'F'
""").show()
print(f"Liquid clustered: {time.time() - start:.2f}s")

##This is critical to understand. There are two very different storage locations:

## Storage Tiers in Databricks

| Location | Survives cluster shutdown? | Use for |
|---|---|---|
| Cluster local disk (`/tmp`, `file:/`) | ❌ No — gone when cluster stops | Truly temporary work |
| DBFS (`/dbfs/`, `dbfs:/`) | ✅ Yes — persistent across clusters | Files, datasets |
| Managed tables (Hive metastore / Unity Catalog) | ✅ Yes — persistent | Tables you query with SQL |
| External cloud storage (S3, ADLS, GCS) | ✅ Yes — most durable | Production data |

run saveAsTable("orders_by_year"), Databricks already stores it persistently in the metastore — it survives cluster restarts. You don't need to do anything extra for tables.

##Saving to a Specific Location (External Tables)
If you want to control where files live:

### Write to a specific DBFS path
df.write.mode("overwrite") \
    .partitionBy("order_year") \
    .save("dbfs:/FileStore/my_data/orders_by_year")

### Or save as table pointing to a specific location
df.write.mode("overwrite") \
    .partitionBy("order_year") \
    .option("path", "dbfs:/FileStore/my_data/orders_by_year") \
    .saveAsTable("orders_by_year_external")

Saving to Cloud Storage (Production Pattern)

### Azure Data Lake
df.write.mode("overwrite") \
    .partitionBy("order_year") \
    .save("abfss://container@account.dfs.core.windows.net/data/orders/")


## Creating RDD

There are 2 ways to create RDD's

### Example: create RDD from Python list or exsisting data using parallelize
nums = [1, 2, 3, 4, 5]
rdd = spark.sparkContext.parallelize(nums) #“Take this local collection and split it into multiple partitions across executors.”

print("Partitions:", rdd.getNumPartitions())

print("First element:", rdd.first())

### From a text file
rdd = spark.sparkContext.textFile("/Volumes/workspace/default/internal/sample.txt")

print("Total lines:", rdd.count())

print("First line:", rdd.first())

Now the code below wont work because serverless dont allow RDD analytics you would need one ckuster to execute the code below

In [0]:
nums = [1, 2, 3, 4, 5]
rdd = spark.sparkContext.parallelize(nums) #“Take this local collection and split it into multiple partitions across executors.”

print("Partitions:", rdd.getNumPartitions())
print("First element:", rdd.first())

In [0]:
3. Transformations (lazy)

# map(func) → apply function to each element

# filter(func) → keep only elements matching condition

# flatMap(func) → like map, but flattens result

rdd = spark.sparkContext.parallelize(["apple banana", "orange", "grape banana"])

# map Applies a function to each element of the RDD and returns a new RDD with the transformed elements.
mapped = rdd.map(lambda x: x.upper())

# filter Keeps only the elements that satisfy a given condition.
filtered = rdd.filter(lambda x: "banana" in x)

# flatMap - Applies a function that returns an iterable (like a list) to each element, then flattens the result.
# Example:
# rdd = sc.parallelize(["hello world", "spark rdd"])
# rdd.flatMap(lambda x: x.split()).collect()
# Output: ['hello', 'world', 'spark', 'rdd']
flatmapped = rdd.flatMap(lambda x: x.split(" "))

print(mapped.collect())    #pulls all the elements of the transformed RDD back to the driver as a list.
print(filtered.collect())   
print(flatmapped.collect()) 


##  Actions (trigger execution)

count() → number of elements

collect() → bring data to driver

take(n) → first n elements

reduce(func) → aggregate

In [0]:
nums = spark.sparkContext.parallelize([1, 2, 3, 4, 5])

print("Count:", nums.count())
print("First:", nums.first())
print("Take 3:", nums.take(3))
print("Sum:", nums.reduce(lambda a, b: a + b))


In [0]:
nums = spark.sparkContext.parallelize(range(1, 101), 5)
print("Partitions:", nums.getNumPartitions())

# Coalesce (reduce partitions)
nums2 = nums.coalesce(2)
print("Partitions after coalesce:", nums2.getNumPartitions())

# Repartition (increase/decrease)
nums3 = nums.repartition(10)
print("Partitions after repartition:", nums3.getNumPartitions())


DataFrames are preferred in production: performance (Catalyst + Tungsten), easier SQL integration, schema awareness, less verbose.

### 📌 When to use RDDs instead of DataFrames

Fine-grained control → You need custom transformations that aren’t easily expressed in SQL/DataFrame APIs.
Example: Complex parsing of nested logs, custom recursive algorithms.

Unstructured / semi-structured data → If the data doesn’t fit into a tabular schema cleanly (before applying schema-on-read).

Low-level operations → When you want to control partitioning, caching, or play with distributed operations explicitly.

Legacy Spark jobs → Older pipelines may still use RDDs for backward compatibility.